In [22]:
import numpy as np
import pandas as pd


In [23]:
# Load in movies dataset from parent directory

movies = pd.read_csv('../ml-32m/movies.csv')
ratings = pd.read_csv('../ml-32m/ratings.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [24]:
# Use this if files are uploaded on google drive:

'''
from google.colab import drive
drive.mount('/content/drive')

movies = pd.read_csv('../ml-32m/movies.csv')
ratings = pd.read_csv('../ml-32m/ratings.csv')
'''

"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nmovies = pd.read_csv('../ml-32m/movies.csv')\nratings = pd.read_csv('../ml-32m/ratings.csv')\n"

In [25]:
# Ratings dataset is too big! Reduce to 400,000 rows (~2500 users)

ratings = ratings.sample(n=400000, random_state=42)

In [26]:
# Merge datasets on movie ID

df = ratings.merge(movies, on='movieId')


# Create user-movie matrix

user_movie_matrix = df.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

/tmp/ipykernel_11178/3392334712.py:8: PerformanceWarning: The following operation may generate 2398061302 cells in the resulting pandas object.
  user_movie_matrix = df.pivot_table(


In [27]:
# Simple recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Sort correlations highest to lowest
    recommendations = corr_df.sort_values(by='correlation', ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [28]:
recommend_movies('Love Actually (2003)')

/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,correlation
title,
Beverly Hills Cop III (1994),1.0
Adaptation (2002),1.0
"40-Year-Old Virgin, The (2005)",1.0
"Matrix Revolutions, The (2003)",1.0
Meet the Parents (2000),1.0
Music and Lyrics (2007),1.0
Bedtime Stories (2008),1.0
City Slickers II: The Legend of Curly's Gold (1994),1.0
Cloud Atlas (2012),1.0


In [29]:
# Slightly more complicated recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Remove NaN values
    corr_df = corr_df.dropna()

    # Count number of ratings per movie
    rating_counts = df.groupby("title")["rating"].count()

    # Add rating counts
    corr_df["num_ratings"] = rating_counts

    # Filter out unpopular movies
    recommendations = corr_df[corr_df["num_ratings"] >= 30].sort_values(
        by="correlation",
        ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [30]:
recommend_movies('Super Troopers (2001)')

/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,correlation,num_ratings
title,,
Billy Madison (1995),1.0,135
Orphan (2009),1.0,35
Philadelphia (1993),1.0,287
Super 8 (2011),1.0,95
"Sound of Music, The (1965)",1.0,226
"Life Aquatic with Steve Zissou, The (2004)",1.0,129
Idiocracy (2006),1.0,82
Mean Girls (2004),1.0,176
Ferris Bueller's Day Off (1986),1.0,393
